In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train_binary = (y_train == 3).astype(np.float32)
y_test_binary = (y_test == 3).astype(np.float32)

x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0


model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


model.summary()

history = model.fit(
    x_train,
    y_train_binary,
    validation_data=(x_test, y_test_binary),
    epochs=10,
    batch_size=64
)

loss, accuracy = model.evaluate(x_test, y_test_binary)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")


plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)
plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(
    history.history['loss'],
    label='Training Loss'
)
plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

model.save('cat_detector_cnn.h5')

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

from tensorflow import keras
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.preprocessing import image_dataset_from_directory

import os
import matplotlib.image as mpimg

In [ ]:
import os

data_path = '/content/dog_vs_cat.zip'

print(f"Extracting dataset from: {data_path}...")
!unzip -o -q "{data_path}" -d /content/
print("Extraction completed. Checking directories:")
print("Exists:", os.path.exists('/content/dog-vs-cat-classification'))

Extracting dataset from: /content/dog_vs_cat.zip...
[/content/dog_vs_cat.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/dog_vs_cat.zip or
        /content/dog_vs_cat.zip.zip, and cannot find /content/dog_vs_cat.zip.ZIP, period.
Extraction completed. Checking directories:
Exists: False


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

cat_dir = os.path.join('dog-vs-cat-classification/cat')
dog_dir = os.path.join('dog-vs-cat-classification/dog')

if not os.path.exists(cat_dir) or not os.path.exists(dog_dir):
    print("Error: The dataset directories do not exist. Please make sure the zip file extracts successfully in the previous cell before running this cell.")
else:
    fig = plt.gcf()
    fig.set_size_inches(16, 16)

    cat_names = os.listdir(cat_dir)
    dog_names = os.listdir(dog_dir)

    pic_index = min(210, len(cat_names), len(dog_names))

    cat_images = [os.path.join(cat_dir, fname)
                  for fname in cat_names[max(0, pic_index-8):pic_index]]
    dog_images = [os.path.join(dog_dir, fname)
                  for fname in dog_names[max(0, pic_index-8):pic_index]]

    for i, img_path in enumerate(cat_images + dog_images):
        sp = plt.subplot(4, 4, i+1)
        sp.axis('Off')

        img = mpimg.imread(img_path)
        plt.imshow(img)

    plt.show()

Error: The dataset directories do not exist. Please make sure the zip file extracts successfully in the previous cell before running this cell.


In [ ]:
base_dir = 'dog-vs-cat-classification'

train_datagen = image_dataset_from_directory(base_dir,
image_size=(200,200),subset='training',seed = 1,validation_split=0.1,batch_size= 32)
test_datagen = image_dataset_from_directory(base_dir,image_size=(200,200),subset='validation',seed = 1,validation_split=0.1,batch_size= 32)

FileNotFoundError: [Errno 2] No such file or directory: 'dog-vs-cat-classification'

In [ ]:
model = tf.keras.models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(200, 200, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.1),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.2),
    layers.BatchNormalization(),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(train_datagen,
          epochs=10,
          validation_data=test_datagen)

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
history_df.loc[:, ['accuracy', 'val_accuracy']].plot()
plt.show()

In [ ]:
def predict_image(image_path):
    img = image.load_img(image_path, target_size=(200, 200))
    plt.imshow(img)
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)

    result = model.predict(img)
    print("Dog" if result >= 0.5 else "Cat")

predict_image('dog-vs-cat-classification/cat/cat.320.jpg')
predict_image('dog-vs-cat-classification/dog/dog.5510.jpg')